In [0]:
# %pip install faker

In [0]:
%run ./config/pipeline_config

## Don't run This as it will again generate raw datasets

In [0]:
# import random
# from faker import Faker
# from pyspark.sql.functions import col, lit, rand, when
# from pyspark.sql.types import *

# # 1. Initialize Faker
# fake = Faker()

# # 3. Define high-volume targets
# NUM_CUSTOMERS = 5000
# NUM_PRODUCTS = 200
# NUM_ORDERS = 10000
# NUM_ORDER_ITEMS = 25000

# print(f"Generating data for paths in: {PATHS['raw']}")

# # --- 1. GENERATE CUSTOMERS ---
# # Schema: customer_id, first_name, last_name, email, signup_date, country 
# cust_list = []
# for i in range(1, NUM_CUSTOMERS + 1):
#     cust_list.append((
#         i, fake.first_name(), fake.last_name(), fake.email(),
#         fake.date_between(start_date='-2y', end_date='today').strftime('%Y-%m-%d'),
#         fake.country()
#     ))
# # Salting for Phase 2 validation: Duplicate ID [cite: 73]
# cust_list.append((1, "Duplicate", "User", "bad_email_format", "2024-01-01", "India"))

# cust_schema = "customer_id INT, first_name STRING, last_name STRING, email STRING, signup_date STRING, country STRING"
# spark.createDataFrame(cust_list, cust_schema).write.mode("overwrite").option("header", "true").csv(f"{PATHS['raw']}/customers.csv")

# # --- 2. GENERATE PRODUCTS ---
# # Schema: product_id, product_name, category, price 
# categories = ['Electronics', 'Furniture', 'Clothing', 'Home Decor', 'Hardware']
# prod_list = []
# for i in range(1, NUM_PRODUCTS + 1):
#     prod_list.append((
#         i, f"{fake.word().capitalize()} {fake.word()}",
#         random.choice(categories),
#         round(random.uniform(10.0, 1500.0), 2)
#     ))
# # Salting for Phase 2: Negative price [cite: 74]
# prod_list.append((999, "Broken Item", "Test", -10.00))

# prod_schema = "product_id INT, product_name STRING, category STRING, price DOUBLE"
# spark.createDataFrame(prod_list, prod_schema).write.mode("overwrite").option("header", "true").csv(f"{PATHS['raw']}/products.csv")

# # --- 3. GENERATE ORDERS ---
# # Schema: order_id, customer_id, order_date, status 
# statuses = ['CREATED', 'SHIPPED', 'DELIVERED', 'CANCELLED']
# orders_df = spark.range(5001, 5001 + NUM_ORDERS).select(
#     col("id").alias("order_id"),
#     (rand() * NUM_CUSTOMERS + 1).cast("int").alias("customer_id"),
#     lit(fake.date_this_year().strftime('%Y-%m-%d')).alias("order_date"),
#     when(rand() < 0.25, statuses[0]).when(rand() < 0.50, statuses[1])
#     .when(rand() < 0.75, statuses[2]).otherwise(statuses[3]).alias("status")
# )
# # Salting for Phase 2: NULL date [cite: 75]
# orders_df = orders_df.withColumn("order_date", when(rand() < 0.05, lit(None)).otherwise(col("order_date")))

# orders_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATHS['raw']}/orders.csv")

# # --- 4. GENERATE ORDER ITEMS ---
# # Schema: order_item_id, order_id, product_id, quantity 
# order_items_df = spark.range(1, NUM_ORDER_ITEMS + 1).select(
#     col("id").alias("order_item_id"),
#     (rand() * NUM_ORDERS + 5001).cast("int").alias("order_id"),
#     (rand() * NUM_PRODUCTS + 1).cast("int").alias("product_id"),
#     (rand() * 10 + 1).cast("int").alias("quantity")
# )
# order_items_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{PATHS['raw']}/order_items.csv")

# print(f"✅ Generated {NUM_CUSTOMERS+1} Customers, {NUM_PRODUCTS+1} Products, {NUM_ORDERS} Orders, and {NUM_ORDER_ITEMS} Items.")

In [0]:
# 1. List files in the raw path to confirm existence
print("Listing files in RAW path:")
display(dbutils.fs.ls(PATHS['raw']))

# 2. Function to load and display sample data
def preview_raw_data(file_name):
    path = f"{PATHS['raw']}/{file_name}.csv"
    print(f"\n--- Previewing: {file_name}.csv ---")
    
    # Reading with header=true and inferSchema=true for verification
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    
    # Displaying schema and top 5 rows
    df.printSchema()
    df.show(10, truncate=False)

# 3. Preview all mandatory datasets
for dataset in ["customers", "products", "orders", "order_items"]:
    preview_raw_data(dataset)